# Annotations, RR intervals and heart rate

Annotation files label events in a record: beats, rhythm changes, noise,
comments. Each annotation has a time (a sample number), a type code with a
mnemonic (`N` for a normal beat, `V` for a premature ventricular
contraction, ...), and optional subtype, channel, number and text fields.

In [ ]:
import numpy as np
import wfdbx

# The sample record 100s, included with wfdb-x, is the first minute of
# record 100 of the MIT-BIH Arrhythmia Database. Read it from the
# package's data directory, wfdbx.sample_data_dir().
wfdbx.wfdbx_set_input_dir(wfdbx.sample_data_dir())

In [ ]:
ann = wfdbx.read_annotations("100s", "atr")
print(ann, "- times in units of 1 /", ann.fs, "s")
for k in range(8):
    print(ann.samples[k], ann.symbols[k], wfdbx.anndesc(ann.codes[k]))

In [ ]:
from collections import Counter
Counter(ann.symbols)

## RR intervals

`isqrs(code)` tells whether an annotation type is a beat (a QRS complex).
The RR intervals are the differences between the times of consecutive
beats.

In [ ]:
is_beat = np.array([bool(wfdbx.isqrs(c)) for c in ann.codes])
beats = ann.samples[is_beat]
rr = np.diff(beats) / ann.fs            # seconds
hr = 60 / rr                            # beats per minute
print(f"{len(beats)} beats, mean RR {rr.mean():.3f} s, "
      f"mean heart rate {hr.mean():.1f} bpm")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:  # the plots are optional
    plt = None
    print("matplotlib is not installed: pip install matplotlib "
          "(or pip install \"wfdb-x[examples]\")")

if plt:
    ecg = wfdbx.read_record("100s", channels=["MLII"], sampto="0:10",
                            physical=True)
    t = np.arange(len(ecg.signals)) / ecg.fs
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5))
    ax1.plot(t, ecg.signals[:, 0], linewidth=0.8)
    shown = ann.samples < len(ecg.signals)
    for s, sym in zip(ann.samples[shown], np.array(ann.symbols)[shown]):
        ax1.annotate(sym, (s / ecg.fs, ecg.signals[s, 0]), color="C3")
    ax1.set_xlabel("time (s)")
    ax1.set_ylabel("MLII")
    ax2.plot(beats[1:] / ann.fs, hr, marker=".")
    ax2.set_xlabel("time (s)")
    ax2.set_ylabel("heart rate (bpm)")
    plt.tight_layout()
    plt.show()

## Selecting a time range

`sampfrom` and `sampto` select annotations by time, as for signals:

In [ ]:
first = wfdbx.read_annotations("100s", "atr", sampto="0:5")
list(zip(first.samples, first.symbols))